# 02 · Building & customising circuits

This notebook is for researchers who want to go beyond the packaged primitives:
the code registry, the circuit-builder vocabulary, reproducing the **Table III**
gate/DEM counts, and getting at the **parity-check matrix** so you can plug in your
own decoder.


In [ ]:
# --- make the tmcbs package importable whether this notebook is launched from
# --- the package directory or from notebooks/ ------------------------------
import os, sys
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_cand, "tmcbs")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import numpy as np
import tmcbs as t
print("tmcbs version", t.__version__)

# Decoder used throughout. "teser" (Tesseract) is the decoder behind the paper's
# main results. If the compiled tesseract_decoder wheel will not load on your
# machine, switch to a pure-Python fallback: "BP-OSD" or "LSD".
DECODER = "teser"


## The code registry

`list_codes()` enumerates every named code; `get_code(name)` (or the `surface_code`
/ `bb_code` factories) instantiates one. The BB codes carry the polynomial
parameters from Supplementary Table 1.


In [ ]:
print(f"{'code':18s} {'n':>4} {'k':>3} {'d':>3} {'k/n':>7}")
for name in t.list_codes():
    c = t.get_code(name)
    print(f"{c.name:18s} {c.n:>4} {c.k:>3} {c.d:>3} {c.k/c.n:>7.3f}")

# A custom BB code from raw exponents (a(x,y)=1+x+y, b(x,y)=1+x^2+y^2):
custom = t.build_bb_code(l=3, m=3, Ax=[0,1], Ay=[1], Bx=[0,2], By=[2], d=4,
                         name="my [[18,4,4]]")
print("\ncustom:", custom, "(n,k,d) =", (custom.n, custom.k, custom.d))


## The circuit-builder vocabulary

Each primitive is assembled in `experimentDefinitionsDoingZs.py` by driving a
circuit builder — `CircuitBuilder` for BB codes, `CircuitBuilderSurface` for
surface codes. They share a method vocabulary:

| method | what it appends |
|--------|-----------------|
| `initQubits(block)` | noisy reset of a code block |
| `extractionRound([blocks], firstPass=...)` | one syndrome-extraction cycle (+ detectors) |
| `prepareEbits(transError)` | create `n` noisy Bell pairs across the link |
| `transversalOp("CX"/"H", [a,b], typeArr=...)` | transversal gate between blocks/ebits |
| `measureEbit0ThenCorrectEbit1()` / `measureEbit1ThenCorrectCB(b)` | LOCC half of a non-local CNOT |
| `measureDataQubits([blocks])` | destructive logical readout |
| `endOfCircuitDetectorsForLogicalMeasurementReadout(b)` / `obsOffset(b, off)` | declare final detectors & logical observables |

`build_for_experiment(code, experiment, p, p_ebit, ...)` is the single dispatch
used by both these notebooks and the MPI runner; `t.Experiment` names the indices.


In [ ]:
for e in t.Experiment:
    print(f"  {int(e)}  {e.name}")

# build_for_experiment dispatches on the enum (or the bare int):
cx = t.build_for_experiment(sc := t.surface_code(3), t.Experiment.NON_LOCAL_CNOT,
                            p=1e-3, p_ebit=1e-3)
print("\nnon-local CNOT via dispatch -> detectors:", cx.num_detectors)


## Reproduce Table III (detectors & error mechanisms)

The paper's Table III reports, per code and primitive, the size of the space-time
parity-check matrix from the STIM DEM: **rows** = detectors, **cols** = distinct
error mechanisms. We can read both straight off the constructed circuits — an
end-to-end check that the circuits match the paper.


In [ ]:
def rows_cols(circ):
    chk, obs, priors = t.parity_check_matrices(circ)  # (detectors x mechanisms), ...
    return chk.shape

# (code, paper rows/cols for non-local CNOT, paper rows/cols for teleportation)
paper = {
    "[[9,1,3]] SC":   ((64, 245),  (64, 238)),
    "[[25,1,5]] SC":  ((192, 847), (192, 832)),
    "[[18,4,4]] BB":  ((144, 1314),(144, 1314)),
    "[[36,4,6]] BB":  ((288, 2628),(288, 2646)),
}
print(f"{'code':16s} {'prim':5s} {'rows(got/paper)':>18} {'cols(got/paper)':>18}")
for name, ((cxr, cxc), (tr, tc)) in paper.items():
    c = t.get_code(name)
    for prim, (pr, pc) in (("cnot", (cxr, cxc)), ("tele", (tr, tc))):
        circ = (t.non_local_cnot(c, 1e-3, 1e-3) if prim == "cnot"
                else t.teleportation(c, 1e-3, 1e-3))
        r, cc = rows_cols(circ)
        ok = "OK" if (r, cc) == (pr, pc) else "DIFF"
        print(f"{name:16s} {prim:5s} {f'{r}/{pr}':>18} {f'{cc}/{pc}':>18}  {ok}")


## Extract the parity-check matrix for your own decoder

`dem_to_check_matrices` returns the check matrix `chk` (detectors × mechanisms),
the observable matrix `obs`, and the per-mechanism prior probabilities. With these
you can decode sampled syndromes with any decoder you like — here a manual BP-OSD
loop, to show the moving parts.


In [ ]:
from ldpc import BpOsdDecoder

code = t.bb_code("[[36,4,6]] BB")
circ = t.non_local_cnot(code, p=1e-3, p_ebit=1e-3)
chk, obs, priors = t.parity_check_matrices(circ)
print("chk (detectors x mechanisms):", chk.shape,
      "| observables:", obs.shape, "| priors:", priors.shape)

bp = BpOsdDecoder(chk, channel_probs=list(priors), max_iter=10000,
                  bp_method="minimum_sum", osd_method="osd_cs", osd_order=7)

det, obs_actual = circ.compile_detector_sampler().sample(2000, separate_observables=True)
fails = 0
for s, o in zip(det.astype(np.uint8), obs_actual):
    ehat = bp.decode(s)
    fails += np.any((obs @ ehat + o) % 2)
print(f"manual BP-OSD: {fails}/2000 logical failures (LER ~ {fails/2000:.2e})")


## Comparing decoders

`estimate_ler` accepts `decoder=` ∈ `{"teser", "BP-OSD", "LSD"}`.
For the BB codes, the search-based **Tesseract** decoder is what brings their
logical performance up to surface-code level (the paper's Supplementary Note 7
shows the gap under BP-OSD). Compare them on one small instance:


In [ ]:
code = t.bb_code("[[36,4,6]] BB")
circ = t.non_local_cnot(code, p=2e-3, p_ebit=2e-3)
for dec in ("teser", "BP-OSD", "LSD"):
    try:
        r = t.estimate_ler(circ, p=2e-3, d=code.d, decoder=dec, target_errors=25)
        print(f"  {dec:8s} -> {r}")
    except Exception as e:
        print(f"  {dec:8s} -> unavailable ({type(e).__name__})")


## Building your own experiment

The packaged primitives live in `tmcbs/experiments.py` and are thin wrappers over
the builder vocabulary above. To prototype a new gadget, copy one of those
functions and re-order the builder calls. As a minimal illustration, the library
already exposes a *local* (same-device, no-ebit) CNOT next to the non-local one —
useful as a baseline.

`custom_noise=<rate>` injects extra
two-qubit depolarising noise after the transversal CNOT — the on-device analogue
of the ebit noise (`p_ebit`) the non-local CNOT pays for:

In [ ]:
code = t.surface_code(5)
local  = t.estimate_ler(t.local_cnot(code, p=1e-3),       p=1e-3, d=code.d,
                        decoder=DECODER, target_errors=40)
noisy  = t.estimate_ler(t.local_cnot(code, p=1e-3, custom_noise=10e-3), p=1e-3, d=code.d,
                        decoder=DECODER, target_errors=40)
nonloc = t.estimate_ler(t.non_local_cnot(code, 1e-3, 10e-3), p=1e-3, d=code.d,
                        decoder=DECODER, target_errors=40)
print("local  CNOT      :", local)
print("local  + custom  :", noisy, " <- custom_noise=10p on the transversal CNOT")
print("non-local        :", nonloc, " <- ebit_noise=10p")